# 🏔️ VAKS — Guia de Estudo: Blockchain & Smart Contracts

Este notebook cobre tudo o que precisas para implementar o **Ledger Service** do projeto VAKS com MetaMask + Avalanche Fuji Testnet.

## 📚 Índice
1. Conceitos Blockchain
2. Wallets, Chaves e Endereços
3. Transações e Gas
4. Solidity & ERC-20 — O Token VAKS
5. Hardhat — Compilar, Testar e Deploy
6. ethers.js — Interagir com o Contrato
7. Integração com o Ledger Service (NestJS)

---
> **Como usar:** Lê a teoria em cada secção e corre as células de código para experimentar. As células marcadas com 🔧 são práticas — modifica-as à vontade.

---
# 1. 🧱 Conceitos Blockchain

## O que é uma Blockchain?

Uma blockchain é uma **base de dados distribuída e imutável**. Em vez de um servidor central, os dados ficam replicados em milhares de nós (computadores) ao redor do mundo.

Cada **bloco** contém:
- Um conjunto de transações
- O hash do bloco anterior (daí a "chain")
- Um timestamp
- Um nonce (número usado na mineração)

```
Bloco 1          Bloco 2          Bloco 3
┌──────────┐    ┌──────────┐    ┌──────────┐
│ hash:    │    │ hash:    │    │ hash:    │
│ 0x1a2b.. │◄───│ prev:    │◄───│ prev:    │
│ txs: ... │    │ 0x1a2b.. │    │ 0x3c4d.. │
└──────────┘    └──────────┘    └──────────┘
```

## Ethereum vs Avalanche

| | Ethereum | Avalanche (C-Chain) |
|---|---|---|
| Linguagem | Solidity | Solidity (EVM compatível) |
| Velocidade | ~15 tx/s | ~4500 tx/s |
| Gas fees | Altas | Baixas |
| Testnet | Sepolia | Fuji |
| Explorer | Etherscan | Snowtrace |

✅ **Para o VAKS usamos Avalanche Fuji Testnet** — mesma linguagem que Ethereum mas muito mais rápido e barato.

In [ ]:
# 🔧 Simulação: como funciona o hash de um bloco
import hashlib
import json
from datetime import datetime

def create_block(index, transactions, previous_hash):
    block = {
        'index': index,
        'timestamp': str(datetime.now()),
        'transactions': transactions,
        'previous_hash': previous_hash,
    }
    block_string = json.dumps(block, sort_keys=True)
    block['hash'] = hashlib.sha256(block_string.encode()).hexdigest()
    return block

# Simula uma chain com 3 blocos
genesis = create_block(0, [], '0000000000000000')
block1 = create_block(1, [{'from': 'Alice', 'to': 'Bob', 'amount': 100}], genesis['hash'])
block2 = create_block(2, [{'from': 'Bob', 'to': 'Campaign#1', 'amount': 50}], block1['hash'])

chain = [genesis, block1, block2]

for block in chain:
    print(f"Bloco #{block['index']}")
    print(f"  Hash:      {block['hash'][:20]}...")
    print(f"  Prev Hash: {block['previous_hash'][:20]}...")
    print(f"  Txs:       {block['transactions']}")
    print()

In [ ]:
# 🔧 Demonstração: imutabilidade — o que acontece se alterares um bloco?
print('=== ANTES da alteração ===')
print(f'Bloco 1 hash: {block1["hash"][:30]}...')
print(f'Bloco 2 prev: {block2["previous_hash"][:30]}...')
print(f'Bloco 2 está válido: {block2["previous_hash"] == block1["hash"]}')

# Altera uma transação no bloco 1 (ataque)
block1_tampered = block1.copy()
block1_tampered['transactions'] = [{'from': 'Alice', 'to': 'Hacker', 'amount': 100}]
block_string = json.dumps({k: v for k, v in block1_tampered.items() if k != 'hash'}, sort_keys=True)
block1_tampered['hash'] = hashlib.sha256(block_string.encode()).hexdigest()

print('\n=== DEPOIS da alteração ===')
print(f'Bloco 1 hash: {block1_tampered["hash"][:30]}...')
print(f'Bloco 2 prev: {block2["previous_hash"][:30]}...')
print(f'Bloco 2 está válido: {block2["previous_hash"] == block1_tampered["hash"]}')
print('\n⛔ A chain está inválida — por isso a blockchain é imutável!')

---
# 2. 🔑 Wallets, Chaves e Endereços

## Como funciona uma wallet?

Uma wallet **não guarda tokens** — guarda as **chaves privadas** que provam que és o dono de um endereço.

```
Chave Privada (256 bits, SECRETA)
       │
       ▼ (criptografia de curva elíptica secp256k1)
Chave Pública
       │
       ▼ (keccak256 hash + últimos 20 bytes)
Endereço: 0x742d35Cc6634C0532925a3b844Bc454e4438f44e
```

## Regras de ouro
- 🔴 **Nunca** partilhes a chave privada
- 🟡 A seed phrase (12/24 palavras) **gera** a chave privada
- 🟢 O endereço pode ser partilhado — é o teu "IBAN" na blockchain

## MetaMask
O MetaMask é uma wallet que vive no browser como extensão. Quando o utilizador quer assinar uma transação, o MetaMask pede confirmação e assina com a chave privada **sem a expor** ao site.

In [ ]:
# Instalar biblioteca de criptografia Ethereum
!pip install eth-account eth-keys -q

In [ ]:
# 🔧 Gerar uma wallet de teste (NUNCA uses isto com dinheiro real)
from eth_account import Account
import secrets

Account.enable_unaudited_hdwallet_features()

# Gera conta com mnemonic
account, mnemonic = Account.create_with_mnemonic()

print('🆕 Nova wallet gerada (apenas para estudo!)')
print(f'Seed phrase:   {mnemonic}')
print(f'Chave privada: {account.key.hex()}')
print(f'Endereço:      {account.address}')
print()
print('⚠️  NUNCA uses esta wallet com fundos reais!')
print('ℹ️  Para o projeto, usa o MetaMask e a Fuji Testnet com AVAX de faucet.')

In [ ]:
# 🔧 Assinar uma mensagem (como o MetaMask faz)
from eth_account.messages import encode_defunct

private_key = account.key
message = 'Autorizo contribuição de 150 VAKS para Campaign #1'

msg = encode_defunct(text=message)
signed = Account.sign_message(msg, private_key=private_key)

print(f'Mensagem:   {message}')
print(f'Assinatura: {signed.signature.hex()[:40]}...')

# Verificar quem assinou
recovered = Account.recover_message(msg, signature=signed.signature)
print(f'Assinado por: {recovered}')
print(f'Correto: {recovered == account.address}')

---
# 3. ⛽ Transações e Gas

## O que é uma transação?

Qualquer operação que **muda o estado** da blockchain é uma transação:
- Enviar AVAX/ETH de A para B
- Chamar uma função de um smart contract (ex: `transfer()`)
- Fazer deploy de um contrato

## O que é Gas?

Gas é a **taxa de processamento** paga aos validadores da rede. Funciona assim:

```
Custo total = Gas Used × Gas Price

Exemplo:
  Transfer ERC-20:  ~65,000 gas
  Gas Price Fuji:   ~25 nAVAX (0.000000025 AVAX)
  Custo total:      ~0.001625 AVAX ≈ quase zero
```

## Estrutura de uma transação

```json
{
  "from":     "0xAlice...",
  "to":       "0xContrato...",
  "value":    0,
  "data":     "0xa9059cbb...",  // ABI encoded: transfer(address, uint256)
  "gas":      65000,
  "gasPrice": 25000000000,
  "nonce":    42,               // Número sequencial para evitar replay attacks
  "chainId":  43113             // Avalanche Fuji
}
```

In [ ]:
# 🔧 Simular o custo de transações no VAKS

operations = {
    'Deploy contrato VAKS': 800_000,
    'mint() — criar tokens': 55_000,
    'transfer() — contribuir para campanha': 65_000,
    'balanceOf() — consultar saldo (READ)': 0,  # Grátis! Não muda estado
    'approve() — autorizar gasto': 46_000,
}

gas_price_navax = 25  # nAVAX (nanoscale AVAX)
avax_price_eur = 22   # preço aproximado AVAX em EUR

print(f'{'Operação':<45} {'Gas':>10} {'AVAX':>12} {'EUR':>10}')
print('-' * 80)

for op, gas in operations.items():
    if gas == 0:
        print(f'{op:<45} {"READ"!s:>10} {"grátis":>12} {"grátis":>10}')
    else:
        avax_cost = (gas * gas_price_navax) / 1e18
        eur_cost = avax_cost * avax_price_eur
        print(f'{op:<45} {gas:>10,} {avax_cost:>12.6f} {eur_cost:>10.4f}€')

---
# 4. 📝 Solidity & ERC-20 — O Token VAKS

## O que é um Smart Contract?

Um smart contract é um **programa que vive na blockchain**. Uma vez deployed, o código é imutável e executa exatamente como escrito — sem intermediários.

## O Standard ERC-20

ERC-20 é o standard para tokens fungíveis (todos os tokens são iguais, como dinheiro). Define estas funções obrigatórias:

```solidity
function totalSupply() returns (uint256)
function balanceOf(address account) returns (uint256)
function transfer(address to, uint256 amount) returns (bool)
function allowance(address owner, address spender) returns (uint256)
function approve(address spender, uint256 amount) returns (bool)
function transferFrom(address from, address to, uint256 amount) returns (bool)
```

## O Contrato VAKS Token

In [ ]:
# O smart contract VAKS completo — copia isto para o teu projeto Hardhat
vaks_contract = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/**
 * @title VAKSToken
 * @dev Token ERC-20 para a plataforma VAKS de crowdfunding.
 * O admin pode fazer mint. Qualquer holder pode transferir.
 */
contract VAKSToken {
    string public name     = "VAKS Token";
    string public symbol   = "VAKS";
    uint8  public decimals = 18;

    address public admin;
    uint256 private _totalSupply;

    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    // Eventos — ficam registados na blockchain para sempre
    event Transfer(address indexed from, address indexed to, uint256 value);
    event Mint(address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);

    // Modificador — só o admin pode chamar funções marcadas
    modifier onlyAdmin() {
        require(msg.sender == admin, "Only admin");
        _;
    }

    constructor() {
        admin = msg.sender; // Quem fizer deploy é o admin
    }

    // --- READ functions (gratuitas) ---

    function totalSupply() public view returns (uint256) {
        return _totalSupply;
    }

    function balanceOf(address account) public view returns (uint256) {
        return _balances[account];
    }

    function allowance(address owner, address spender) public view returns (uint256) {
        return _allowances[owner][spender];
    }

    // --- WRITE functions (custam gas) ---

    /**
     * @dev Cria novos tokens. Apenas o admin (Ledger Service) pode chamar.
     * Usado quando um utilizador compra VAKS ou recebe recompensa.
     */
    function mint(address to, uint256 amount) public onlyAdmin {
        require(to != address(0), "Cannot mint to zero address");
        _totalSupply        += amount;
        _balances[to]       += amount;
        emit Mint(to, amount);
        emit Transfer(address(0), to, amount);
    }

    /**
     * @dev Transfere tokens. Usado quando um utilizador contribui para uma campanha.
     */
    function transfer(address to, uint256 amount) public returns (bool) {
        require(_balances[msg.sender] >= amount, "Insufficient balance");
        require(to != address(0), "Cannot transfer to zero address");
        _balances[msg.sender] -= amount;
        _balances[to]         += amount;
        emit Transfer(msg.sender, to, amount);
        return true;
    }

    /**
     * @dev Autoriza um spender (ex: contrato de campanha) a gastar tokens em nome do owner.
     */
    function approve(address spender, uint256 amount) public returns (bool) {
        _allowances[msg.sender][spender] = amount;
        emit Approval(msg.sender, spender, amount);
        return true;
    }

    /**
     * @dev Transfere tokens em nome de outro utilizador (requer approve prévio).
     */
    function transferFrom(address from, address to, uint256 amount) public returns (bool) {
        require(_balances[from] >= amount, "Insufficient balance");
        require(_allowances[from][msg.sender] >= amount, "Allowance exceeded");
        _balances[from]               -= amount;
        _balances[to]                 += amount;
        _allowances[from][msg.sender] -= amount;
        emit Transfer(from, to, amount);
        return true;
    }
}
'''
print(vaks_contract)

In [ ]:
# 🔧 Simular o comportamento do contrato em Python
# (Útil para entender a lógica antes de escrever em Solidity)

class VAKSToken:
    def __init__(self, admin):
        self.admin         = admin
        self.total_supply  = 0
        self.balances      = {}
        self.allowances    = {}
        self.events        = []

    def mint(self, caller, to, amount):
        assert caller == self.admin, 'Only admin'
        assert to != '0x0', 'Cannot mint to zero address'
        self.total_supply          += amount
        self.balances[to]           = self.balances.get(to, 0) + amount
        self.events.append(f'Mint: {amount} VAKS → {to}')

    def transfer(self, caller, to, amount):
        assert self.balances.get(caller, 0) >= amount, 'Insufficient balance'
        self.balances[caller] -= amount
        self.balances[to]      = self.balances.get(to, 0) + amount
        self.events.append(f'Transfer: {amount} VAKS {caller} → {to}')

    def balance_of(self, account):
        return self.balances.get(account, 0)


# Simula o fluxo completo do VAKS
token = VAKSToken(admin='0xAdmin')

# Admin faz mint para utilizadores
token.mint('0xAdmin', '0xAlice', 1000)
token.mint('0xAdmin', '0xBob',    500)

# Alice contribui para Campaign #1
token.transfer('0xAlice', '0xCampaign1', 150)
token.transfer('0xAlice', '0xCampaign1',  50)

# Bob contribui
token.transfer('0xBob', '0xCampaign1', 200)

print('=== Estado Final ===')
for address, balance in token.balances.items():
    print(f'  {address}: {balance} VAKS')

print(f'\nTotal supply: {token.total_supply} VAKS')
print(f'Campaign #1 recebeu: {token.balance_of("0xCampaign1")} VAKS')

print('\n=== Histórico de Eventos ===')
for event in token.events:
    print(f'  {event}')

# Teste de erro
print('\n=== Teste de erro ===')
try:
    token.transfer('0xBob', '0xAlice', 9999)  # Bob não tem saldo
except AssertionError as e:
    print(f'  ✅ Erro apanhado: {e}')

---
# 5. 🔨 Hardhat — Compilar, Testar e Deploy

## O que é o Hardhat?

Hardhat é o framework de desenvolvimento para smart contracts. Faz o mesmo que NestJS faz para o teu backend — estrutura o projeto, corre testes e faz deploy.

## Setup do projeto Hardhat para VAKS

Corre estes comandos no teu terminal (não no Colab):

```bash
# Na raiz do projeto VAKS
mkdir ledger-contracts && cd ledger-contracts
npm init -y
npm install --save-dev hardhat @nomicfoundation/hardhat-toolbox dotenv
npx hardhat init  # Escolhe: TypeScript project
```

## Estrutura do projeto

```
ledger-contracts/
├── contracts/
│   └── VAKSToken.sol        ← O contrato Solidity
├── scripts/
│   └── deploy.ts            ← Script de deploy
├── test/
│   └── VAKSToken.test.ts    ← Testes
├── hardhat.config.ts        ← Configuração (redes, etc.)
└── .env                     ← Chave privada do deployer (NUNCA commitares)
```

In [ ]:
# Os ficheiros de configuração prontos a usar

hardhat_config = '''
// hardhat.config.ts
import { HardhatUserConfig } from 'hardhat/config';
import '@nomicfoundation/hardhat-toolbox';
import * as dotenv from 'dotenv';
dotenv.config();

const config: HardhatUserConfig = {
  solidity: '0.8.20',
  networks: {
    fuji: {
      url: 'https://api.avax-test.network/ext/bc/C/rpc',
      chainId: 43113,
      accounts: [process.env.DEPLOYER_PRIVATE_KEY!],
    },
    hardhat: {},  // Rede local para testes
  },
  etherscan: {
    apiKey: {
      avalancheFujiTestnet: process.env.SNOWTRACE_API_KEY!,
    },
  },
};

export default config;
'''

deploy_script = '''
// scripts/deploy.ts
import { ethers } from 'hardhat';

async function main() {
  const [deployer] = await ethers.getSigners();
  console.log('Deploying with:', deployer.address);

  const VAKSToken = await ethers.getContractFactory('VAKSToken');
  const token = await VAKSToken.deploy();
  await token.waitForDeployment();

  const address = await token.getAddress();
  console.log('VAKSToken deployed to:', address);
  console.log('Snowtrace:', `https://testnet.snowtrace.io/address/${address}`);
}

main().catch(console.error);
'''

test_file = '''
// test/VAKSToken.test.ts
import { ethers } from 'hardhat';
import { expect } from 'chai';

describe('VAKSToken', () => {
  async function deploy() {
    const [admin, alice, bob] = await ethers.getSigners();
    const VAKSToken = await ethers.getContractFactory('VAKSToken');
    const token = await VAKSToken.deploy();
    return { token, admin, alice, bob };
  }

  it('admin pode fazer mint', async () => {
    const { token, alice } = await deploy();
    await token.mint(alice.address, 1000);
    expect(await token.balanceOf(alice.address)).to.equal(1000);
  });

  it('non-admin não pode fazer mint', async () => {
    const { token, alice } = await deploy();
    await expect(token.connect(alice).mint(alice.address, 1000))
      .to.be.revertedWith('Only admin');
  });

  it('transfer funciona com saldo suficiente', async () => {
    const { token, admin, alice, bob } = await deploy();
    await token.mint(alice.address, 500);
    await token.connect(alice).transfer(bob.address, 150);
    expect(await token.balanceOf(alice.address)).to.equal(350);
    expect(await token.balanceOf(bob.address)).to.equal(150);
  });

  it('transfer falha sem saldo', async () => {
    const { token, alice, bob } = await deploy();
    await expect(token.connect(alice).transfer(bob.address, 100))
      .to.be.revertedWith('Insufficient balance');
  });
});
'''

print('=== hardhat.config.ts ===')
print(hardhat_config)
print('=== scripts/deploy.ts ===')
print(deploy_script)
print('=== test/VAKSToken.test.ts ===')
print(test_file)

In [ ]:
# Comandos Hardhat — referência rápida
comandos = {
    'Compilar contratos':           'npx hardhat compile',
    'Correr testes (rede local)':   'npx hardhat test',
    'Correr testes com gas report': 'REPORT_GAS=true npx hardhat test',
    'Deploy na Fuji Testnet':       'npx hardhat run scripts/deploy.ts --network fuji',
    'Verificar no Snowtrace':       'npx hardhat verify --network fuji <CONTRACT_ADDRESS>',
    'Console interativo (local)':   'npx hardhat console',
}

print('📋 Comandos Hardhat')
print('=' * 60)
for desc, cmd in comandos.items():
    print(f'\n{desc}:')
    print(f'  $ {cmd}')

print('\n\n💡 Antes do deploy na Fuji precisas de:')
print('  1. AVAX de testnet — faucet: https://faucet.avax.network')
print('  2. DEPLOYER_PRIVATE_KEY no .env (wallet de desenvolvimento)')
print('  3. SNOWTRACE_API_KEY — regista em https://snowtrace.io')

---
# 6. ⚡ ethers.js — Interagir com o Contrato

## O que é o ethers.js?

ethers.js é a biblioteca TypeScript/JavaScript para interagir com contratos Ethereum/Avalanche. É o que o teu **Ledger Service NestJS** vai usar para:
- Chamar `mint()` quando um utilizador recebe VAKS
- Chamar `transfer()` quando há uma contribuição
- Chamar `balanceOf()` para consultar saldos
- Escutar eventos `Transfer` e `Mint` em tempo real

## Conceitos chave

| Conceito | Descrição |
|---|---|
| **Provider** | Ligação à rede (Fuji RPC) — apenas lê |
| **Signer** | Provider + chave privada — pode escrever |
| **Contract** | Instância do contrato com ABI |
| **ABI** | Interface do contrato (gerado pelo Hardhat) |

In [ ]:
# O código completo do Ledger Service para o teu projeto
# Copia para Backend/ledger-service/src/ledger/ledger.service.ts

ledger_service = '''
// Backend/ledger-service/src/ledger/ledger.service.ts
import { Injectable, Logger, OnModuleInit } from '@nestjs/common';
import { ethers } from 'ethers';
import { PrismaService } from '../prisma/prisma.service';

// ABI mínimo — apenas as funções que usamos
const VAKS_ABI = [
  'function mint(address to, uint256 amount) external',
  'function transfer(address to, uint256 amount) external returns (bool)',
  'function balanceOf(address account) external view returns (uint256)',
  'function totalSupply() external view returns (uint256)',
  'event Transfer(address indexed from, address indexed to, uint256 value)',
  'event Mint(address indexed to, uint256 value)',
];

@Injectable()
export class LedgerService implements OnModuleInit {
  private readonly logger = new Logger(LedgerService.name);
  private provider: ethers.JsonRpcProvider;
  private signer: ethers.Wallet;
  private contract: ethers.Contract;

  constructor(private readonly prisma: PrismaService) {}

  async onModuleInit() {
    // Ligar à Avalanche Fuji Testnet
    this.provider = new ethers.JsonRpcProvider(
      process.env.AVALANCHE_RPC_URL ?? 'https://api.avax-test.network/ext/bc/C/rpc'
    );

    // Admin wallet para assinar transações (mint, etc.)
    this.signer = new ethers.Wallet(process.env.ADMIN_PRIVATE_KEY!, this.provider);

    // Instanciar o contrato
    this.contract = new ethers.Contract(
      process.env.VAKS_CONTRACT_ADDRESS!,
      VAKS_ABI,
      this.signer,
    );

    this.logger.log(`Ledger Service connected. Contract: ${process.env.VAKS_CONTRACT_ADDRESS}`);
    this.listenToEvents();
  }

  // Consultar saldo on-chain (READ — gratuito)
  async getBalance(walletAddress: string): Promise<string> {
    const balance = await this.contract.balanceOf(walletAddress);
    return ethers.formatUnits(balance, 18); // Converte de wei para VAKS
  }

  // Criar tokens para um utilizador (WRITE — custa gas)
  async mint(toAddress: string, amount: number): Promise<string> {
    const amountWei = ethers.parseUnits(amount.toString(), 18);
    const tx = await this.contract.mint(toAddress, amountWei);
    const receipt = await tx.wait();

    // Guardar no Ledger local
    await this.prisma.ledgerEntry.create({
      data: {
        userId: 0, // Associar ao userId real
        txHash: receipt.hash,
        amount,
        operation: 'MINT',
        blockNumber: receipt.blockNumber,
      },
    });

    this.logger.log(`Minted ${amount} VAKS to ${toAddress} — tx: ${receipt.hash}`);
    return receipt.hash;
  }

  // Transferir tokens (WRITE)
  async transfer(fromAddress: string, toAddress: string, amount: number): Promise<string> {
    // Nota: em produção, o from assina com o MetaMask no frontend
    // O backend só trata transferências admin (ex: contribuições automáticas)
    const amountWei = ethers.parseUnits(amount.toString(), 18);
    const tx = await this.contract.transfer(toAddress, amountWei);
    const receipt = await tx.wait();

    await this.prisma.ledgerEntry.create({
      data: {
        userId: 0,
        txHash: receipt.hash,
        amount,
        operation: 'TRANSFER',
        blockNumber: receipt.blockNumber,
        metadata: { from: fromAddress, to: toAddress },
      },
    });

    return receipt.hash;
  }

  // Detalhes de uma transação
  async getTransaction(txHash: string) {
    const tx = await this.provider.getTransaction(txHash);
    const receipt = await this.provider.getTransactionReceipt(txHash);
    return {
      hash: txHash,
      blockNumber: receipt?.blockNumber,
      status: receipt?.status === 1 ? 'SUCCESS' : 'FAILED',
      explorerUrl: `https://testnet.snowtrace.io/tx/${txHash}`,
    };
  }

  // Escutar eventos on-chain em tempo real
  private listenToEvents() {
    this.contract.on('Mint', (to, value, event) => {
      this.logger.log(`[EVENT] Mint: ${ethers.formatUnits(value, 18)} VAKS → ${to}`);
    });

    this.contract.on('Transfer', (from, to, value, event) => {
      this.logger.log(`[EVENT] Transfer: ${ethers.formatUnits(value, 18)} VAKS ${from} → ${to}`);
    });
  }
}
'''
print(ledger_service)

---
# 7. 🔗 Integração Completa — Fluxo VAKS

## Como tudo se liga

```
Frontend (React + MetaMask)
         │
         │  1. Utilizador clica "Contribuir 150 VAKS"
         │  2. MetaMask pede confirmação
         │  3. MetaMask assina a transação
         ▼
API Gateway (NestJS)
         │
         │  4. Verifica JWT do utilizador
         ▼
Campaign Service ──► Wallet Service
         │                  │
         │           5. Debita saldo off-chain
         │                  │
         ▼                  ▼
Ledger Service
         │
         │  6. Chama transfer() no smart contract
         │  7. Aguarda confirmação on-chain
         │  8. Guarda txHash no LedgerEntry
         ▼
Avalanche Fuji Testnet
         │
         │  9. Transação confirmada
         ▼
Snowtrace Explorer
  https://testnet.snowtrace.io/tx/0x...
```

## Variáveis de ambiente necessárias

```env
# .env do ledger-service
AVALANCHE_RPC_URL=https://api.avax-test.network/ext/bc/C/rpc
ADMIN_PRIVATE_KEY=0x...    # Wallet do backend (NUNCA expor)
VAKS_CONTRACT_ADDRESS=0x... # Endereço do contrato após deploy
SNOWTRACE_API_KEY=...       # Para verificação do contrato
```

## Checklist de implementação

- [ ] Escrever `VAKSToken.sol`
- [ ] Configurar Hardhat com rede Fuji
- [ ] Escrever e passar testes
- [ ] Obter AVAX de testnet no faucet
- [ ] Deploy na Fuji Testnet
- [ ] Verificar no Snowtrace
- [ ] Implementar Ledger Service (NestJS)
- [ ] Integrar com Wallet Service
- [ ] Frontend: conectar MetaMask
- [ ] Frontend: mostrar saldo e histórico

In [ ]:
# 🔧 Simulação completa do fluxo de contribuição VAKS

class VAKSPlatform:
    def __init__(self):
        self.token      = VAKSToken(admin='0xLedgerService')
        self.campaigns  = {}
        self.wallets    = {}   # userId → wallet address
        self.tx_log     = []

    def register_user(self, user_id, wallet_address, initial_vaks=0):
        self.wallets[user_id] = wallet_address
        if initial_vaks > 0:
            self.token.mint('0xLedgerService', wallet_address, initial_vaks)
            self.tx_log.append(f'[MINT] User {user_id} recebeu {initial_vaks} VAKS')

    def create_campaign(self, campaign_id, owner_id, goal):
        self.campaigns[campaign_id] = {
            'owner': owner_id, 'goal': goal,
            'current': 0, 'wallet': f'0xCampaign{campaign_id}'
        }

    def contribute(self, user_id, campaign_id, amount):
        campaign = self.campaigns[campaign_id]
        user_wallet = self.wallets[user_id]
        campaign_wallet = campaign['wallet']

        self.token.transfer(user_wallet, campaign_wallet, amount)
        campaign['current'] += amount

        tx_hash = f'0x{hash((user_id, campaign_id, amount)):016x}'
        self.tx_log.append(f'[TRANSFER] User {user_id} → Campaign {campaign_id}: {amount} VAKS | tx: {tx_hash}')

        if campaign['current'] >= campaign['goal']:
            self.tx_log.append(f'[EVENT] 🎉 goal.reached Campaign {campaign_id}!')

        return tx_hash


# Simula o fluxo completo
platform = VAKSPlatform()

# Regista utilizadores com saldo inicial
platform.register_user(1, '0xAlice',   initial_vaks=1000)
platform.register_user(2, '0xBob',     initial_vaks=500)
platform.register_user(3, '0xCarlos',  initial_vaks=750)

# Cria campanha
platform.create_campaign('camp1', owner_id=1, goal=500)

# Contribuições
platform.contribute(2, 'camp1', 200)
platform.contribute(3, 'camp1', 150)
platform.contribute(2, 'camp1', 150)  # Esta atinge a meta!

print('=== Saldos Finais ===')
for user_id, wallet in platform.wallets.items():
    balance = platform.token.balance_of(wallet)
    print(f'  User {user_id} ({wallet}): {balance} VAKS')

for cid, camp in platform.campaigns.items():
    balance = platform.token.balance_of(camp['wallet'])
    print(f'  Campaign {cid}: {balance} VAKS (meta: {camp["goal"]})')

print('\n=== Histórico On-Chain ===')
for entry in platform.tx_log:
    print(f'  {entry}')

---
## 🎯 Próximos Passos

Agora que entendes os conceitos, a ordem de implementação para o Issue #9:

### 1. Preparar ambiente
```bash
mkdir ledger-contracts && cd ledger-contracts
npm init -y
npm install --save-dev hardhat @nomicfoundation/hardhat-toolbox dotenv
npx hardhat init
```

### 2. Escrever o contrato
- Copia o `VAKSToken.sol` da Secção 4 para `contracts/VAKSToken.sol`

### 3. Testar localmente
```bash
npx hardhat test
```

### 4. Obter AVAX de testnet
- Vai a https://faucet.avax.network
- Adiciona a rede Fuji no MetaMask (chainId: 43113)

### 5. Deploy
```bash
npx hardhat run scripts/deploy.ts --network fuji
```

### 6. Implementar o Ledger Service
- Cria `Backend/ledger-service/` com a estrutura NestJS
- Copia o `LedgerService` da Secção 6
- Liga ao Wallet Service via eventos

---
*Notebook criado para o projeto VAKS — Issue #9 Ledger Service*